# Sweep prediction summary analyzer

Loads the `sweep_predict_summary.csv` produced by any of the three sweep-predict scripts and answers the questions you'd actually run after a sweep:

1. **Top-N by prediction quality** (default `accuracy@iou0.50`) and by `val_loss_final`.
2. **Does training loss predict prediction quality?** Scatter of `val_loss_final` vs the primary prediction metric, colored by optimizer / scheduler.
3. **Per-IoU sweep** — how each model's primary metric degrades from IoU=0.3 → 0.5 → 0.7.
4. **Optimizer × scheduler grid** — heatmap of mean primary metric for each combo.
5. **LR sensitivity** — primary metric vs learning rate, grouped by optimizer.
6. **Per-IoU full-metric table** for the top-K runs (precision / recall / f1 / accuracy / PQ).

Edit the `CSV_PATH` / `PRIMARY_METRIC` / `PRIMARY_IOU` constants in the first config cell and Run-All.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Point at whichever sweep you want to analyze.
#   StarDist:       unet_sweeps        →  scripts/model_prediction/stardist_sweeps/sweep_predict_summary.csv
#   U-Net:          unet_sweeps        →  scripts/model_prediction/unet_sweeps/sweep_predict_summary.csv
#   ROI → StarDist: stardist_roi_sweeps/sweep_predict_summary.csv
CSV_PATH = Path("unet_sweeps/sweep_predict_summary.csv")

# What metric / IoU to treat as primary throughout the notebook.
# Anything ``matching_dataset`` returned at that IoU is available:
# precision / recall / f1 / accuracy / panoptic_quality /
# mean_matched_score / mean_true_score.
PRIMARY_METRIC = "accuracy"
PRIMARY_IOU = 0.5

TOP_K = 10  # used by every "top-K" table / plot below
IOUS = (0.3, 0.5, 0.7)
PRIMARY_COL = f"{PRIMARY_METRIC}@iou{PRIMARY_IOU:.2f}"

# Skip optimizers / schedulers / experiment-name substrings entirely
# (e.g. failed lr=10 runs you've already binned). Empty list = keep all.
SKIP_EXPERIMENT_SUBSTRINGS: list[str] = []

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"{CSV_PATH} not found — point CSV_PATH at one of the three "
        f"sweep_predict_summary.csv outputs."
    )
df = pd.read_csv(CSV_PATH)
for sub in SKIP_EXPERIMENT_SUBSTRINGS:
    df = df[~df["experiment"].str.contains(sub, na=False)]

# Learning rate sometimes came in as a string like "1.0e-3" from the
# experiment-name tag parser; coerce to float so it sorts numerically.
if "learning_rate" in df.columns:
    df["learning_rate"] = pd.to_numeric(df["learning_rate"], errors="coerce")

if PRIMARY_COL not in df.columns:
    raise KeyError(
        f"{PRIMARY_COL} not in CSV columns — available metric columns: "
        + ", ".join(c for c in df.columns if "@iou" in c)
    )

print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Columns: {list(df.columns)}")
print()
print(f"Primary column: {PRIMARY_COL!r}")
print(f"  range: [{df[PRIMARY_COL].min():.4f}, {df[PRIMARY_COL].max():.4f}]  "
      f"non-null: {df[PRIMARY_COL].notna().sum()}/{len(df)}")
df.head()

## 1. Top-N rankings
Best by prediction quality (descending), best by training val_loss (ascending). The two often disagree — that's what the next cell visualises.

In [ ]:
tag_cols = [c for c in ("experiment", "optimizer", "learning_rate", "scheduler") if c in df.columns]
metric_cols_at_primary = [
    c for c in df.columns if c.endswith(f"@iou{PRIMARY_IOU:.2f}") and c != PRIMARY_COL
]
train_cols = [c for c in ("val_loss_final", "train_loss_final", "epochs_done") if c in df.columns]

print(f"── Best by {PRIMARY_COL} (top {TOP_K}) ──")
best_pred = (
    df.dropna(subset=[PRIMARY_COL])
      .sort_values(PRIMARY_COL, ascending=False)
      .head(TOP_K)
)
display(best_pred[tag_cols + [PRIMARY_COL] + train_cols])

if "val_loss_final" in df.columns:
    print(f"\n── Best by val_loss_final ↓ (top {TOP_K}) ──")
    best_train = (
        df.dropna(subset=["val_loss_final"])
          .sort_values("val_loss_final")
          .head(TOP_K)
    )
    display(best_train[tag_cols + ["val_loss_final", PRIMARY_COL]])

## 2. Training loss vs prediction quality
If `val_loss_final` is well-correlated with the primary metric, the cheap-and-fast training proxy is reliable — you can trust it for early stopping. If the cloud is scattered, the prediction-quality sweep is the only honest source of truth.

In [ ]:
if "val_loss_final" not in df.columns:
    print("val_loss_final not in CSV — skipping correlation plot")
else:
    sub = df.dropna(subset=["val_loss_final", PRIMARY_COL])
    if sub.empty:
        print("No rows with both val_loss_final and primary metric — skipping")
    else:
        rho_p = sub["val_loss_final"].corr(sub[PRIMARY_COL], method="pearson")
        rho_s = sub["val_loss_final"].corr(sub[PRIMARY_COL], method="spearman")

        fig, ax = plt.subplots(figsize=(8, 5))
        opts = sorted(sub["optimizer"].dropna().unique().tolist()) if "optimizer" in sub.columns else [None]
        palette = plt.get_cmap("tab10").colors
        for i, opt in enumerate(opts):
            mask = sub["optimizer"] == opt if opt is not None else slice(None)
            ax.scatter(
                sub.loc[mask, "val_loss_final"],
                sub.loc[mask, PRIMARY_COL],
                color=palette[i % len(palette)],
                label=str(opt),
                s=55, alpha=0.85, edgecolors="black", linewidth=0.4,
            )
        ax.set_xlabel("val_loss_final")
        ax.set_ylabel(PRIMARY_COL)
        ax.set_title(
            f"Training proxy vs prediction quality\n"
            f"Pearson ρ = {rho_p:.3f}    Spearman ρ = {rho_s:.3f}"
        )
        ax.grid(True, alpha=0.3)
        if opts and opts[0] is not None:
            ax.legend(title="optimizer", loc="best", fontsize=8)
        fig.tight_layout()
        plt.show()

## 3. Per-IoU degradation
How sharply each model's primary metric falls as the IoU bar rises. A model that's `accuracy=0.7` at IoU=0.3 but `0.05` at IoU=0.7 has the right cells in roughly the right place but the boundaries are sloppy; flat curves = sharp boundaries.

In [ ]:
iou_cols = [f"{PRIMARY_METRIC}@iou{iou:.2f}" for iou in IOUS]
iou_cols_present = [c for c in iou_cols if c in df.columns]
if len(iou_cols_present) < 2:
    print(f"Need ≥ 2 IoU columns for the primary metric; found {iou_cols_present}")
else:
    # Plot top-K by PRIMARY_COL only — otherwise the chart is unreadable.
    top = (
        df.dropna(subset=[PRIMARY_COL])
          .sort_values(PRIMARY_COL, ascending=False)
          .head(TOP_K)
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    xs = [float(c.split("@iou")[1]) for c in iou_cols_present]
    for _, row in top.iterrows():
        ys = [row[c] for c in iou_cols_present]
        ax.plot(xs, ys, marker="o", linewidth=1.4, alpha=0.85,
                label=row.get("experiment", "?"))
    ax.set_xlabel("IoU threshold")
    ax.set_ylabel(PRIMARY_METRIC)
    ax.set_title(f"Top {len(top)} runs — {PRIMARY_METRIC} per IoU")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=7, ncol=1)
    fig.tight_layout()
    plt.show()

## 4. Optimizer × scheduler grid
Mean and best primary metric per (optimizer, scheduler) combo. Tells you which corners of the hyperparameter space are worth re-sweeping at finer granularity.

In [ ]:
if {"optimizer", "scheduler"}.issubset(df.columns):
    pivot_mean = df.pivot_table(
        index="optimizer", columns="scheduler",
        values=PRIMARY_COL, aggfunc="mean",
    )
    pivot_max = df.pivot_table(
        index="optimizer", columns="scheduler",
        values=PRIMARY_COL, aggfunc="max",
    )
    pivot_n = df.pivot_table(
        index="optimizer", columns="scheduler",
        values=PRIMARY_COL, aggfunc="count",
    )

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, mat, title in (
        (axes[0], pivot_mean, f"Mean {PRIMARY_COL}"),
        (axes[1], pivot_max,  f"Best {PRIMARY_COL}"),
    ):
        im = ax.imshow(mat.values, cmap="viridis", aspect="auto")
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=20, ha="right")
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index)
        ax.set_title(title)
        for i, opt in enumerate(mat.index):
            for j, sched in enumerate(mat.columns):
                v = mat.iloc[i, j]
                if pd.notna(v):
                    ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                            color="white" if v < (mat.values.max() * 0.55) else "black",
                            fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

    print("\nRun counts per (optimizer, scheduler):")
    display(pivot_n.fillna(0).astype(int))
else:
    print("optimizer / scheduler columns missing — skipping grid")

## 5. LR sensitivity per optimizer
Log-x scatter of primary metric vs learning rate, one trace per optimizer. The shape of the curve tells you whether the sweep grid is hitting the optimum or you need to refine.

In [ ]:
if {"learning_rate", "optimizer"}.issubset(df.columns):
    sub = df.dropna(subset=["learning_rate", PRIMARY_COL])
    fig, ax = plt.subplots(figsize=(8, 5))
    palette = plt.get_cmap("tab10").colors
    for i, opt in enumerate(sorted(sub["optimizer"].dropna().unique())):
        s = sub[sub["optimizer"] == opt].sort_values("learning_rate")
        ax.plot(s["learning_rate"], s[PRIMARY_COL], marker="o",
                color=palette[i % len(palette)], label=opt, linewidth=1.4)
    ax.set_xscale("log")
    ax.set_xlabel("learning rate (log)")
    ax.set_ylabel(PRIMARY_COL)
    ax.set_title(f"{PRIMARY_COL} vs learning rate, per optimizer")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(title="optimizer", loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("learning_rate / optimizer columns missing — skipping LR plot")

## 6. Full per-IoU metric table for top-K runs
Wide view: every available metric × every available IoU for the top-K models by primary metric. Useful when you need to argue "this model has high accuracy but bad PQ" without scrolling through the raw CSV.

In [ ]:
iou_metric_cols = [c for c in df.columns if re.match(r".+@iou\d\.\d{2}$", c)]
top = (
    df.dropna(subset=[PRIMARY_COL])
      .sort_values(PRIMARY_COL, ascending=False)
      .head(TOP_K)
)
display(top[tag_cols + train_cols + sorted(iou_metric_cols)])